In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier

# --- Load & Preprocess ---
dataset_path = '../Email Dataset/Clean_Emails_Dataset.csv'
df = pd.read_csv(dataset_path)
df = df.dropna(subset=['Email Text', 'Email Type'])
df['label_num'] = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

X = df['Email Text'].astype(str)
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Folder to save models ---
os.makedirs('../Models', exist_ok=True)

# --- TF-IDF Vectorizer (shared by LR and NB) ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

with open('../Models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("Saved: tfidf_vectorizer.pkl")

# --- Logistic Regression ---
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)

with open('../Models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print("Saved: lr_model.pkl")

# --- Naive Bayes ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

with open('../Models/nb_model.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
print("Saved: nb_model.pkl")

# --- Word2Vec + Random Forest ---
X_train_tokens = [text.split() for text in X_train]

w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=1, workers=4)
w2v_model.save('../Models/w2v_model.model')
print("Saved: w2v_model.model")

def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_train_tokens])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_w2v, y_train)

with open('../Models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("Saved: rf_model.pkl")

print("\nAll models trained and saved to ../Models/")